1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    hamming_loss,
    roc_auc_score
)

import joblib

2. Mount Google Drive and Load the Preprocessed Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train = pd.read_csv("/content/drive/MyDrive/Data_Science/models/train_cleaned.csv")

print(train.shape)

train.head()

(159547, 10)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,clean_text,word_count
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,explanation why the edits made under my userna...,47
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,d aww ! he match this background colour i am s...,17
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,hey man i am really not trying to edit war it ...,44
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,more i can not make any real suggestion on imp...,116
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,you sir are my hero any chance you remember wh...,15


3. Prepare Features, Labels, and Split the Dataset

In [ ]:
target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

X = train["clean_text"]

y = train[target_columns]

print(y.head())

   toxic  severe_toxic  obscene  threat  insult  identity_hate
0      0             0        0       0       0              0
1      0             0        0       0       0              0
2      0             0        0       0       0              0
3      0             0        0       0       0              0
4      0             0        0       0       0              0


In [ ]:
stratify_labels = (y.sum(axis=1) > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_labels
)

print("Training Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])

Training Samples : 127637
Testing Samples  : 31910


4. Extract TF-IDF Features

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)

(127637, 50000)


5. Train the Logistic Regression Model and Generate Predictions

In [ ]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        class_weight="balanced"
    )
)

model.fit(X_train_tfidf, y_train)

print("Training Completed!")

Training Completed!


In [ ]:
y_pred = model.predict(X_test_tfidf)

y_prob = model.predict_proba(X_test_tfidf)

6. Evaluate Overall Model Performance

In [ ]:
print("="*50)

print("Hamming Loss")
print(hamming_loss(y_test, y_pred))

print("="*50)

print("Micro Precision")
print(precision_score(y_test, y_pred, average="micro"))

print("="*50)

print("Micro Recall")
print(recall_score(y_test, y_pred, average="micro"))

print("="*50)

print("Micro F1 Score")
print(f1_score(y_test, y_pred, average="micro"))

print("="*50)

print("Macro F1 Score")
print(f1_score(y_test, y_pred, average="macro"))

print("="*50)

print("ROC-AUC")
print(roc_auc_score(y_test, y_prob, average="macro"))

Hamming Loss
0.028481144886660398
Micro Precision
0.5700276111587166
Micro Recall
0.8646735990756788
Micro F1 Score
0.6870947380501521
Macro F1 Score
0.5629039600558303
ROC-AUC
0.9806265845292117


7. Generate Classification Report for Each Toxicity Label

In [ ]:
for label in target_columns:

    print("="*70)

    print(label.upper())

    print(classification_report(
        y_test[label],
        y_pred[:, target_columns.index(label)]
    ))

TOXIC
              precision    recall  f1-score   support

           0       0.99      0.95      0.97     28860
           1       0.66      0.86      0.75      3050

    accuracy                           0.95     31910
   macro avg       0.82      0.91      0.86     31910
weighted avg       0.95      0.95      0.95     31910

SEVERE_TOXIC
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     31614
           1       0.27      0.84      0.41       296

    accuracy                           0.98     31910
   macro avg       0.63      0.91      0.70     31910
weighted avg       0.99      0.98      0.98     31910

OBSCENE
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     30255
           1       0.68      0.89      0.77      1655

    accuracy                           0.97     31910
   macro avg       0.84      0.93      0.88     31910
weighted avg       0.98      0.97      0.97    

8. Save the Trained Model and TF-IDF Vectorizer

In [ ]:
joblib.dump(model, "logistic_model.pkl")

joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
samples = [
    "Have a wonderful day.",
    "I will kill you.",
    "You are the dumbest person ever.",
    "Thank you for your support.",
    "i love it."
]

sample_tfidf = tfidf.transform(samples)

pred = model.predict(sample_tfidf)

results = pd.DataFrame(pred, columns=target_columns)

results.insert(0, "Comment", samples)

results

,Comment,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Have a wonderful day.,0,0,0,0,0,0
1,I will kill you.,1,1,1,1,0,0
2,You are the dumbest person ever.,1,0,0,0,1,1
3,Thank you for your support.,0,0,0,0,0,0
4,lets love.,0,0,0,0,0,0


## Logistic Regression Results

Hamming Loss : 0.0285

Micro Precision : 0.5700

Micro Recall : 0.8647

Micro F1 Score : 0.6871

Macro F1 Score : 0.5629

ROC-AUC Score : 0.9806

Observations

• High recall indicates the model successfully identifies most toxic comments.

• Lower precision suggests some non-toxic comments are incorrectly classified as toxic.

• Excellent ROC-AUC (0.9806) shows strong discriminative ability.

• Performance is lower for severe_toxic, threat and identity_hate because of extreme class imbalance.